# KWISMO — Modèle B : Entraînement NLP & Auto-Catégorisation

Ce notebook permet d'exécuter le pipeline complet du **Modèle B** :
1. **Nettoyage des données** (`src.data.clean`) sur le jeu de données `data/raw/kwismo_data/messages.jsonl`.
2. **Augmentation synthétique** (`src.data.augment`) en Franglais/Pidgin local.
3. **Prétraitement & Normalisation** (`src.models.model_b.preprocess`) du franglais/pidgin et auto-catégorisation dynamique.
4. **Entraînement & Sauvegarde** (`src.models.model_b.train`) du modèle NLP et du modèle de repli TF-IDF (priorité à `model_b_augmented.jsonl`).
5. **Validation par des exemples de test** (vérification sur des cas de fraude réels).

In [11]:
# Détection de l'environnement d'exécution
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB:
    ENV_NAME = "Google Colab"
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive
            print("⚡ Connexion automatique à Google Drive...")
            drive.mount('/content/drive')
        except Exception as err:
            print(f"⚠️ Montage Google Drive recommandé : {err}")
elif ON_KAGGLE:
    ENV_NAME = "Kaggle Notebooks"
else:
    ENV_NAME = "Local"

print(f"Environnement de calcul détecté : {ENV_NAME}")

Environnement de calcul détecté : Local


In [12]:
# Configuration du dossier de travail sur Cloud (Colab / Kaggle) et Local
if ON_COLAB:
    PROJECT_DIR = Path("/content/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /content/kwismo
    else:
        !git -C /content/kwismo fetch && git -C /content/kwismo reset --hard origin/main
elif ON_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /kaggle/working/kwismo
    else:
        !git -C /kaggle/working/kwismo fetch && git -C /kaggle/working/kwismo reset --hard origin/main
else:
    # Résolution robuste de la racine kwismo-ai en local
    current = Path.cwd()
    PROJECT_DIR = current
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            PROJECT_DIR = candidate
            break
        elif (candidate / "kwismo-ai" / "src").exists():
            PROJECT_DIR = candidate / "kwismo-ai"
            break

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
print("Dossier racine du projet kwismo-ai :", PROJECT_DIR)

Dossier racine du projet kwismo-ai : C:\Users\PC\Desktop\kwismo\kwismo-ai


In [13]:
# Installation de Python 3.13 et création du venv isolé (.venv313)
VENV_DIR = PROJECT_DIR / ".venv313"

if ON_COLAB or ON_KAGGLE:
    python313_bin = VENV_DIR / "bin" / "python"
    if not python313_bin.exists():
        print("Installation de Python 3.13 et création de l'environnement .venv313...")
        !apt-get update -y
        !apt-get install -y software-properties-common
        !add-apt-repository -y ppa:deadsnakes/ppa
        !apt-get update -y
        !apt-get install -y python3.13 python3.13-venv python3.13-dev
        !python3.13 -m venv {VENV_DIR}
        !{VENV_DIR}/bin/pip install --upgrade pip
        !{VENV_DIR}/bin/pip install -r requirements.txt
    PYTHON_BIN = str(python313_bin)
else:
    PYTHON_BIN = sys.executable

def run_module(module: str) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a échoué (code {result.returncode})")
    else:
        print(result.stdout)

ver_proc = subprocess.run([PYTHON_BIN, "--version"], capture_output=True, text=True)
print("Interprète Python configuré :", PYTHON_BIN)
print("Version vérifiée :", ver_proc.stdout.strip() or ver_proc.stderr.strip())

Interprète Python configuré : c:\Users\PC\AppData\Local\Programs\Python\Python313\python.exe
Version vérifiée : Python 3.13.0


## 1. Nettoyage du Jeu de Données

Exécution de `src.data.clean` pour filtrer le bruit web et structurer le jeu de données propre dans `data/processed/model_b_clean.jsonl`.

In [14]:
run_module("src.data.clean")

Lecture du jeu de donnÃ©es brut depuis : C:\Users\PC\Desktop\kwismo\kwismo-ai\data\raw\kwismo_data\messages.jsonl
Nettoyage terminÃ© : 504 extraits de fraude nettoyÃ©s sauvegardÃ©s dans C:\Users\PC\Desktop\kwismo\kwismo-ai\data\processed\model_b_clean.jsonl



## 2. Augmentation du dataset en Franglais/Pidgin

Exécution de `src.data.augment` pour générer des phrases et rendre les données plus pertinentes dans `data/processed/model_b_augmented.jsonl`.

In [15]:
run_module("src.data.augment")

Augmentation terminÃ©e : 1188 extraits sauvegardÃ©s dans C:\Users\PC\Desktop\kwismo\kwismo-ai\data\processed\model_b_augmented.jsonl



## 3. Entraînement du Modèle B

Exécution de `src.models.model_b.train` pour entraîner le modèle et mettre à jour le registre `models/registry.json`. (Sélectionne prioritairement `model_b_augmented.jsonl`).

In [16]:
run_module("src.models.model_b.train")

Standard Output:

Standard Error:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\PC\Desktop\kwismo\kwismo-ai\src\models\model_b\train.py", line 12, in <module>
    import joblib
ModuleNotFoundError: No module named 'joblib'



RuntimeError: src.models.model_b.train a échoué (code 1)

## 4. Test de Validation du Modèle B

Test du Modèle B sur des exemples de fraude réels pour valider la détection et la catégorisation.

In [ ]:
from src.models.model_b.preprocess import categorize_description, process_report_batch

# Exemple 1 : Faux SMS de transfert reçu
ex_1 = "Mobile Money, vous avez reçu 75 000 FCFA de Achiri (237677620953). Tapez *126# pour valider."
# Exemple 2 : Faux agent prétendant débloquer un compte contre un code PIN
ex_2 = "Bonjour je suis agent Orange Money, votre compte présente un problème. Envoyez votre code secret PIN au 694000000."

test_batch = [
    {"id_signalement": "sig_test_001", "description": ex_1},
    {"id_signalement": "sig_test_002", "description": ex_2},
]

cache = {}
results = process_report_batch(test_batch, cache)

print("=== Résultats du Test de Validation ===")
for sig_id, cat in results.items():
    print(f"ID: {sig_id} ➔ Catégorie détectée: {cat}")